# Week 3 — QLoRA Fine-Tuning

**Self-contained. Runs on Colab GPU. Reuses the proven Week-2 execution harness.**

Goal: fine-tune **Qwen2.5-Coder-7B-Instruct** with **QLoRA** and report a clean
**before (zero-shot) vs after (QLoRA)** comparison across **all four** code-intelligence
directions, so it lines up cell-for-cell with Vamsi's results.

| # | Task | Direction | Source dataset | Eval metric |
|---|------|-----------|----------------|-------------|
| 1 | NL → PL1 | English spec → **Python** | HumanEval (native) | pass@1 (real unit tests) |
| 2 | NL → PL2 | English spec → **Java** | HumanEval-X (native) | pass@1 (real `java_test`) |
| 3 | Code → NL | Python → **English** | HumanEval (docstring = gold) | ROUGE-L F1 |
| 4 | PL1 → PL2 | Python → **Java** | HumanEval-X (native) | pass@1 (real `java_test`) |

**Why this design**
- All four tasks ride on the **same 164 problems** (HumanEval ≡ HumanEval-X), so a **single
  problem-ID split** governs every task → no cross-task leakage (a problem held out for one
  direction is held out for all of them).
- Training targets are **ground-truth** where it exists (native Python/Java) and
  **execution-validated** (`flag=True`) where Week-2 generated it (MBPP→Java augmentation).
- Metrics match Vamsi exactly (pass@1 for code, ROUGE-L F1 for NL) so the two notebooks compose.

**Before running:** Runtime → Change runtime type → **GPU** (A100 / L4 ideal; T4 works but slow).


> **⚡ Runtime-safe (restart-resilient).** All heavy outputs — zero-shot scores, the trained
> adapter, and QLoRA scores — are checkpointed to **Google Drive**, so a Colab disconnect or
> crash no longer forces a from-scratch rerun. On a **T4**, use the **two-phase flow** in
> Section 11: run the zero-shot eval → **Restart runtime** (frees VRAM) → train → eval →
> compare. Cell 1.1 will refuse to run if Drive is not mounted (that prevents silent data loss).

## 1. Setup

In [ ]:
# 1.1 — Drive mount + project dir  (persistent storage is REQUIRED for the restart flow)
import os
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')  # complete the auth popup; raises if it can't mount
    # Persistent project folder on your Drive. Override with the CODEGEN_PROJECT_DIR env var.
    PROJECT_DIR = Path(os.environ.get(
        'CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/lora_finetune'))
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    # Fail loud if we ended up on ephemeral /content — a crash there loses everything.
    assert '/content/drive/' in str(PROJECT_DIR), (
        f'PROJECT_DIR={PROJECT_DIR} is NOT on Google Drive. Outputs would be lost on a '
        f'runtime restart. Fix the mount / CODEGEN_PROJECT_DIR before continuing.')
else:
    PROJECT_DIR = Path('.').resolve()

print('IN_COLAB =', IN_COLAB)
print('PROJECT_DIR =', PROJECT_DIR)

Mounted at /content/drive
IN_COLAB = True
PROJECT_DIR = /content/drive/MyDrive/lora_finetune


In [ ]:
# 1.2 — install JDK 17 (for Java pass@1) + Python deps
import subprocess, sys

def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=False)

# Java toolchain for run_java()
sh('apt-get -qq install -y openjdk-17-jdk-headless > /dev/null 2>&1')
sh('java -version')

# QLoRA stack. Pinned-ish to versions known to play together on Colab.
sh(f'{sys.executable} -m pip -q install -U '
   '"transformers>=4.44" "peft>=0.12" "accelerate>=0.33" '
   '"bitsandbytes>=0.43" "datasets>=2.20" "rouge-score" "huggingface_hub"')
print('deps installed.')

$ apt-get -qq install -y openjdk-17-jdk-headless > /dev/null 2>&1
$ java -version
$ /usr/bin/python3 -m pip -q install -U "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.43" "datasets>=2.20" "rouge-score" "huggingface_hub"
deps installed.


In [ ]:
# 1.3 — config
import os
from pathlib import Path

SEED        = int(os.environ.get('CODEGEN_SEED', '13'))      # team standard
MODEL_SIZE  = os.environ.get('CODEGEN_MODEL', '1.5b')          # '1.5b' for a fast sanity run
N_TEST      = int(os.environ.get('CODEGEN_NTEST', '33'))     # match Vamsi's held-out size
USE_MBPP_AUG = os.environ.get('CODEGEN_MBPP_AUG', '1') == '1'  # extra Py->Java / NL->Java train data

# Which tasks to train+eval. Drop any to run a subset.
TASKS = ['nl2py', 'nl2java', 'code2nl', 'py2java']

# Which tasks the MBPP augmentation feeds. Code->NL intentionally omitted: MBPP's `nl` is a
# terse command, not the descriptive paragraph Code->NL needs as a target.
MBPP_AUG_TASKS = ['nl2py', 'nl2java', 'py2java']

DATA_DIR    = Path(os.environ.get('CODEGEN_DATA_DIR', str(PROJECT_DIR)))
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DATA_DIR / 'lora_adapter_mbppaug'

MODELS = {
    '1.5b': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '7b':   'Qwen/Qwen2.5-Coder-7B-Instruct',
}
MODEL_ID = MODELS[MODEL_SIZE]

# QLoRA hyper-params (mirrors the standard r=8 recipe Vamsi used)
LORA = dict(r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                            'gate_proj', 'up_proj', 'down_proj'])
TRAIN = dict(epochs=3, lr=2e-4, batch=1, grad_accum=16, max_len=1024, warmup_ratio=0.03)

DECODING_GREEDY = {'do_sample': False, 'max_new_tokens': 512}
EXEC_TIMEOUT_S  = 15

print(f'Model={MODEL_ID}  Seed={SEED}  N_TEST={N_TEST}  MBPP_aug={USE_MBPP_AUG}')
print(f'Tasks={TASKS}')
print(f'MBPP_AUG_TASKS={MBPP_AUG_TASKS}')
print(f'Results -> {RESULTS_DIR}')
print(f'Adapter -> {ADAPTER_DIR}')

Model=Qwen/Qwen2.5-Coder-1.5B-Instruct  Seed=13  N_TEST=33  MBPP_aug=True
Tasks=['nl2py', 'nl2java', 'code2nl', 'py2java']
MBPP_AUG_TASKS=['nl2py', 'nl2java', 'py2java']
Results -> /content/drive/MyDrive/lora_finetune/results
Adapter -> /content/drive/MyDrive/lora_finetune/lora_adapter_aditi_mbppaug


In [ ]:
# 1.4 — GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

CUDA available: True
GPU: Tesla T4
VRAM: 15.5 GB free / 15.6 GB total


## 2. Execution Sandbox  *(reused verbatim from Week 2)*

These are the **hard gate** for pass@1: generated code is compiled and run against the real
test harness. Identical to `week2_standalone.ipynb` so results are directly comparable.

In [ ]:
# 2.1 — Python sandbox
import subprocess, sys, tempfile
from pathlib import Path

def run_python(full_program: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'solution.py'
        path.write_text(full_program, encoding='utf-8')
        try:
            res = subprocess.run(
                [sys.executable, str(path)],
                capture_output=True, text=True, timeout=timeout,
                env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
            )
            if res.returncode == 0:
                return {'passed': True, 'error': None}
            err = (res.stderr or res.stdout).strip().splitlines()[-1:]
            return {'passed': False, 'error': '\n'.join(err) or 'nonzero exit'}
        except subprocess.TimeoutExpired:
            return {'passed': False, 'error': f'timeout >{timeout}s'}
        except Exception as e:
            return {'passed': False, 'error': f'runner: {e!r}'}

In [ ]:
# 2.2 — Java sandbox (-ea enables `assert`)
import re as _re

_CLASS_NAME_RE = _re.compile(r'public\s+class\s+(\w+)')
_JAVA_STD_IMPORTS = (
    'import java.util.*;\n'
    'import java.util.stream.*;\n'
    'import java.util.regex.*;\n'
    'import java.lang.*;\n'
    'import java.math.*;\n'
)

def _inject_imports(src: str) -> str:
    head = src.lstrip()
    if head.startswith('package '):
        nl = head.find('\n')
        return head[:nl+1] + _JAVA_STD_IMPORTS + head[nl+1:]
    return _JAVA_STD_IMPORTS + src

def run_java(solution: str, test_source: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    if not solution or not test_source:
        return {'passed': False, 'error': 'missing solution or test block'}
    classes = _CLASS_NAME_RE.findall(test_source)
    main_class = classes[0] if classes else 'Main'
    solution    = _inject_imports(solution)
    test_source = _inject_imports(test_source)
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        (tmp_path / 'Solution.java').write_text(solution, encoding='utf-8')
        (tmp_path / f'{main_class}.java').write_text(test_source, encoding='utf-8')
        try:
            comp = subprocess.run(
                ['javac', '-d', str(tmp_path),
                 str(tmp_path / 'Solution.java'),
                 str(tmp_path / f'{main_class}.java')],
                capture_output=True, text=True, timeout=timeout)
            if comp.returncode != 0:
                return {'passed': False, 'error': 'compile: ' + (comp.stderr or '').strip()[:400]}
            run = subprocess.run(['java', '-ea', '-cp', str(tmp_path), main_class],
                                 capture_output=True, text=True, timeout=timeout)
            if run.returncode == 0:
                return {'passed': True, 'error': None}
            tail = (run.stderr or run.stdout).strip().splitlines()[-3:]
            return {'passed': False, 'error': '\n'.join(tail) or 'nonzero exit'}
        except subprocess.TimeoutExpired:
            return {'passed': False, 'error': f'timeout >{timeout}s'}
        except Exception as e:
            return {'passed': False, 'error': f'runner: {e!r}'}

# smoke test
_sol = 'public class Solution { public static int add(int a, int b){ return a+b; } }'
_tst = ('public class Main { public static void main(String[] a){ '
        'assert Solution.add(2,3)==5; } }')
print('Java sandbox OK:', run_java(_sol, _tst))

Java sandbox OK: {'passed': True, 'error': None}


## 3. Generation & Block Extractors  *(reused from Week 2)*

In [ ]:
# 3.1 — fenced-block extractors
import re

def _strip_fences(text: str, lang_hints: tuple) -> str:
    s = text.strip()
    open_re = re.compile(r'```(?:' + '|'.join(lang_hints) + r')?\s*\n?', re.IGNORECASE)
    m = open_re.search(s)
    if not m:
        return (s + '\n') if s else ''
    body = s[m.end():]
    close = re.search(r'```', body)
    body = body[:close.start()] if close else body
    return (body.rstrip() + '\n') if body.rstrip() else ''

def extract_python_body(text: str) -> str:
    return _strip_fences(text, ('python', 'py'))

def extract_java_body(text: str) -> str:
    return _strip_fences(text, ('java',))

def strip_markdown(text: str) -> str:
    """For Code->NL: remove any code fences, keep prose only."""
    return re.sub(r'```[a-zA-Z]*\n.*?```', '', text, flags=re.S).strip()

assert extract_python_body('```python\ndef f(): pass\n```') == 'def f(): pass\n'
assert extract_java_body('```java\npublic class S {}\n```') == 'public class S {}\n'
print('Extractors OK.')

Extractors OK.


In [ ]:
# 3.2 — chat-template generation (single example at a time, like Week 2)
import torch

def generate_text(tokenizer, model, user_prompt: str, max_new_tokens: int = 512) -> str:
    msgs = [{'role': 'user', 'content': user_prompt}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

## 4. Build the unified problem set + the single shared split

We merge HumanEval (native NL + Python + python tests) with HumanEval-X (native Java +
`java_test`) on the shared problem number. Every task draws its fields from this one merged
record, and **one** problem-ID split (seed 13) defines train vs. test for *all* tasks.

In [ ]:
# 4.1 — HumanEval-X loader (reused from Week 2)
import gzip, json, random
from huggingface_hub import hf_hub_download

def _hex_split(lang: str) -> list:
    REPO = 'THUDM/humaneval-x'
    candidates = [
        f'data/{lang}/data/humaneval.jsonl',
        f'data/{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/humaneval_{lang}.jsonl.gz',
        f'{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/data/humaneval.jsonl.gz',
    ]
    last_err = None
    for filename in candidates:
        try:
            path = hf_hub_download(repo_id=REPO, filename=filename, repo_type='dataset')
            opener = gzip.open if filename.endswith('.gz') else open
            with opener(path, 'rt', encoding='utf-8') as f:
                return [json.loads(line) for line in f if line.strip()]
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Failed to load HumanEval-X "{lang}": {last_err!r}')

In [ ]:
# 4.2 — docstring helper + merged problem records
import ast

def py_docstring(prompt_or_src: str) -> str:
    """Extract the NL docstring from a HumanEval prompt (signature + docstring)."""
    src = prompt_or_src
    try:
        tree = ast.parse(src)
    except SyntaxError:
        # prompt has no body -> add a pass so it parses
        try:
            tree = ast.parse(src.rstrip() + '\n    pass\n')
        except Exception:
            return ''
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            doc = ast.get_docstring(node)
            if doc:
                return doc.strip()
    return ''

def strip_py_docstring(src: str) -> str:
    """Remove docstrings so Code->NL can't read the answer off the code."""
    try:
        tree = ast.parse(src)
    except Exception:
        return src
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            body = getattr(node, 'body', [])
            if (body and isinstance(body[0], ast.Expr)
                    and isinstance(body[0].value, ast.Constant)
                    and isinstance(body[0].value.value, str)):
                node.body = body[1:] or [ast.Pass()]
    try:
        return ast.unparse(tree)
    except Exception:
        return src

In [ ]:
# 4.3 — load + merge HumanEval (native) with HumanEval-X (Java)
from datasets import load_dataset

def load_merged_problems():
    try:
        he = load_dataset('openai/openai_humaneval', split='test')
    except Exception:
        he = load_dataset('openai_humaneval', split='test', trust_remote_code=True)
    by_id_he = {}
    for r in he:
        k = r['task_id'].split('/')[-1]
        by_id_he[k] = {
            'nl':            py_docstring(r['prompt']) or r['prompt'].strip(),
            'py_prompt':     r['prompt'],
            'py_canonical':  r['prompt'] + r['canonical_solution'],
            'py_solution':   r['canonical_solution'],
            'py_test':       r['test'],
            'py_entry':      r['entry_point'],
        }
    py_hex = {r['task_id'].split('/')[-1]: r for r in _hex_split('python')}
    ja_hex = {r['task_id'].split('/')[-1]: r for r in _hex_split('java')}

    problems = []
    for k in sorted(by_id_he, key=lambda x: int(x) if x.isdigit() else 1e9):
        if k not in ja_hex:
            continue
        rj = ja_hex[k]
        rec = dict(by_id_he[k])
        rec.update({
            'id':            f'HumanEval/{k}',
            'num':           k,
            'java_decl':     rj.get('declaration', rj['prompt']),
            'java_canonical': rj['prompt'] + rj['canonical_solution'],
            'java_test':     rj['test'],
        })
        problems.append(rec)
    return problems

problems = load_merged_problems()
print(f'Merged problems (HE ∩ HEX): {len(problems)}')

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

humaneval.jsonl:   0%|          | 0.00/343k [00:00<?, ?B/s]

humaneval.jsonl:   0%|          | 0.00/475k [00:00<?, ?B/s]

Merged problems (HE ∩ HEX): 164


In [ ]:
# 4.4 — the ONE shared split (seed 13). Same held-out IDs for every task.
def split_problems(problems, n_test=N_TEST, seed=SEED):
    idx = list(range(len(problems)))
    random.Random(seed).shuffle(idx)
    test_idx = set(idx[:n_test])
    train = [problems[i] for i in idx[n_test:]]
    test  = [problems[i] for i in idx[:n_test]]
    return train, test

train_problems, test_problems = split_problems(problems)
test_ids = sorted(p['num'] for p in test_problems)
print(f'train={len(train_problems)}  test={len(test_problems)}')
print('Held-out test problem numbers:', test_ids)
print('>> Share this list with Vamsi: identical test IDs == apples-to-apples comparison.')

train=131  test=33
Held-out test problem numbers: ['1', '100', '107', '11', '119', '12', '125', '126', '127', '13', '130', '131', '134', '139', '148', '15', '158', '29', '31', '39', '40', '42', '6', '60', '64', '69', '79', '82', '85', '96', '97', '98', '99']
>> Share this list with Vamsi: identical test IDs == apples-to-apples comparison.


In [ ]:
# 4.5 — optional MBPP -> Java augmentation (train-only, no leakage)
import csv, pandas as pd
csv.field_size_limit(10**7)

mbpp_aug = []
if USE_MBPP_AUG:
    mbpp_csv = RESULTS_DIR / f'extended_mbpp_seed{SEED}.csv'
    if not mbpp_csv.exists():
        raise FileNotFoundError(
            f'USE_MBPP_AUG is True but {mbpp_csv} was not found. '
            f'Upload extended_mbpp_seed{SEED}.csv to {RESULTS_DIR} before running this cell.')
    df = pd.read_csv(mbpp_csv, usecols=['problem_id', 'nl', 'python', 'java', 'flag'])
    java_ok_mask = df['flag'].astype(str).str.strip().str.lower() == 'true'
    for (_, r), java_ok in zip(df.iterrows(), java_ok_mask):
        nl = str(r.get('nl', '')).strip()
        if not nl:
            continue
        mbpp_aug.append({
            'id': f"mbpp/{r['problem_id']}",
            'nl': nl,
            'python': str(r.get('python', '')).strip(),
            'java': str(r.get('java', '')).strip(),
            'java_ok': bool(java_ok),
        })
    assert len(mbpp_aug) > 0, f'Loaded {mbpp_csv} but found 0 usable rows.'
    n_java_ok = sum(1 for m in mbpp_aug if m['java_ok'])
    print(f'MBPP augmentation rows loaded: {len(mbpp_aug)} total, {n_java_ok} with java_ok=True')
else:
    print('MBPP augmentation disabled.')

MBPP augmentation rows loaded: 257 total, 216 with java_ok=True


## 5. Task prompts + training examples

One prompt builder per direction (adapted from Week 2's proven templates). Training examples
are `(instruction_prompt, target)` pairs; we train only on the **target** tokens (prompt masked).

In [ ]:
# 5.1 — prompt builders (one per task)
def p_nl2py(nl, signature):
    return (f"Implement the following specification in Python.\n"
            f"Return ONLY the complete function inside a single ```python``` block. No prose, no tests.\n\n"
            f"# Specification\n{nl}\n\n# Required signature\n```python\n{signature}\n```")

def p_nl2java(nl, java_decl):
    return (f"Implement the following specification in Java.\n"
            f"Return ONLY one ```java``` block: `public class Solution` with the required public static method.\n\n"
            f"# Specification\n{nl}\n\n# Required Java declaration\n```java\n{java_decl}\n```")

def p_code2nl(py_code):
    return (f"Read the Python function and write a SINGLE-PARAGRAPH natural-language specification "
            f"(<=120 words) describing what it does: input types, output type, and edge cases.\n"
            f"Return only the paragraph, no code, no markdown.\n\n"
            f"# Python\n```python\n{py_code}\n```")

def p_py2java(py_code, java_decl):
    return (f"Translate the Python function below to Java.\n"
            f"Return ONE ```java``` block: `public class Solution` with the translated method as public static. "
            f"Match the required Java signature exactly.\n\n"
            f"# Python\n```python\n{py_code}\n```\n\n# Required Java declaration\n```java\n{java_decl}\n```")

def py_signature(py_prompt):
    """First `def ...:` line from a HumanEval prompt."""
    for line in py_prompt.splitlines():
        if line.strip().startswith('def '):
            return line.strip()
    return 'def solution(*args):'

In [ ]:
# 5.2 — assemble training examples for the requested tasks
def build_train_examples(train_problems, mbpp_aug, tasks, mbpp_tasks=None):
    if mbpp_tasks is None:
        mbpp_tasks = tasks
    ex = []
    for p in train_problems:
        if 'nl2py' in tasks:
            ex.append((p_nl2py(p['nl'], py_signature(p['py_prompt'])),
                       f"```python\n{p['py_canonical'].strip()}\n```"))
        if 'nl2java' in tasks:
            ex.append((p_nl2java(p['nl'], p['java_decl']),
                       f"```java\n{p['java_canonical'].strip()}\n```"))
        if 'code2nl' in tasks:
            code_in = strip_py_docstring(p['py_canonical'])
            ex.append((p_code2nl(code_in), p['nl'].strip()))
        if 'py2java' in tasks:
            ex.append((p_py2java(p['py_canonical'], p['java_decl']),
                       f"```java\n{p['java_canonical'].strip()}\n```"))

    # MBPP augmentation: nl2py uses python (ungated); nl2java + py2java require java_ok.
    # Code->NL intentionally excluded even if present in mbpp_tasks: MBPP's `nl` is a terse
    # command, not the descriptive paragraph Code->NL needs, and using it as a target risks
    # regressing the Code->NL score.
    n_nl2py = n_nl2java = n_py2java = 0
    for m in mbpp_aug:
        if 'nl2py' in mbpp_tasks and m['python']:
            ex.append((p_nl2py(m['nl'], py_signature(m['python'])),
                       f"```python\n{m['python'].strip()}\n```"))
            n_nl2py += 1
        if 'nl2java' in mbpp_tasks and m['java_ok']:
            ex.append((p_nl2java(m['nl'], '// translate to a public class Solution'),
                       f"```java\n{m['java'].strip()}\n```"))
            n_nl2java += 1
        if 'py2java' in mbpp_tasks and m['java_ok']:
            ex.append((p_py2java(m['python'], '// translate to a public class Solution'),
                       f"```java\n{m['java'].strip()}\n```"))
            n_py2java += 1

    print(f'MBPP pairs added -> nl2py: {n_nl2py}  nl2java: {n_nl2java}  py2java: {n_py2java}  '
          f'(code2nl excluded from MBPP)')
    random.Random(SEED).shuffle(ex)
    return ex

train_examples = build_train_examples(train_problems, mbpp_aug, TASKS, MBPP_AUG_TASKS)
print(f'Total training examples: {len(train_examples)}')
print('--- sample prompt ---'); print(train_examples[0][0][:300])
print('--- sample target ---'); print(train_examples[0][1][:200])

NameError: name 'mbpp_aug' is not defined

## 6. Load base model (4-bit) + tokenizer

In [ ]:
# 6.1 — 4-bit NF4 load (same quantization as Week 2)
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if MODEL_SIZE == '7b':
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
model.eval()
print('Base model loaded:', MODEL_ID)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct


## 7. Evaluation harness (used for BOTH zero-shot and QLoRA)

`evaluate_all` runs every requested task over the held-out test set and returns the metric per
task. We call it **once before training** (zero-shot = base model) and **once after** (QLoRA).
Per-example predictions are saved for inspection / viva evidence.

In [ ]:
# 7.1 — metrics
from rouge_score import rouge_scorer
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rougeL_f1(ref, hyp):
    if not ref or not hyp:
        return 0.0
    return _rouge.score(ref, hyp)['rougeL'].fmeasure

In [ ]:
# 7.2 — per-task eval over the held-out test set
from tqdm.auto import tqdm
import json

def evaluate_all(model, tokenizer, test_problems, tasks, tag='zero_shot'):
    rows, scores = [], {}

    def add(task, pid, metric, value, passed=None, pred=''):
        rows.append({'tag': tag, 'task': task, 'problem_id': pid,
                     'metric': metric, 'value': value,
                     'passed': passed, 'prediction': pred[:2000]})

    if 'nl2py' in tasks:
        ok = 0
        for p in tqdm(test_problems, desc=f'{tag}:nl2py'):
            raw = generate_text(tokenizer, model, p_nl2py(p['nl'], py_signature(p['py_prompt'])), 512)
            gen = extract_python_body(raw)
            full = gen + '\n' + p['py_test'] + f"\ncheck({p['py_entry']})\n"
            res = run_python(full) if gen else {'passed': False}
            ok += int(bool(res.get('passed')))
            add('nl2py', p['id'], 'pass@1', float(bool(res.get('passed'))), res.get('passed'), gen)
        scores['nl2py'] = ok / max(1, len(test_problems))

    if 'nl2java' in tasks:
        ok = 0
        for p in tqdm(test_problems, desc=f'{tag}:nl2java'):
            raw = generate_text(tokenizer, model, p_nl2java(p['nl'], p['java_decl']), 900)
            gen = extract_java_body(raw)
            res = run_java(gen, p['java_test']) if gen else {'passed': False}
            ok += int(bool(res.get('passed')))
            add('nl2java', p['id'], 'pass@1', float(bool(res.get('passed'))), res.get('passed'), gen)
        scores['nl2java'] = ok / max(1, len(test_problems))

    if 'py2java' in tasks:
        ok = 0
        for p in tqdm(test_problems, desc=f'{tag}:py2java'):
            raw = generate_text(tokenizer, model, p_py2java(p['py_canonical'], p['java_decl']), 900)
            gen = extract_java_body(raw)
            res = run_java(gen, p['java_test']) if gen else {'passed': False}
            ok += int(bool(res.get('passed')))
            add('py2java', p['id'], 'pass@1', float(bool(res.get('passed'))), res.get('passed'), gen)
        scores['py2java'] = ok / max(1, len(test_problems))

    if 'code2nl' in tasks:
        tot = 0.0
        for p in tqdm(test_problems, desc=f'{tag}:code2nl'):
            code_in = strip_py_docstring(p['py_canonical'])
            raw = generate_text(tokenizer, model, p_code2nl(code_in), 400)
            pred = strip_markdown(raw)
            f1 = rougeL_f1(p['nl'].strip(), pred)
            tot += f1
            add('code2nl', p['id'], 'rougeL_f1', f1, None, pred)
        scores['code2nl'] = tot / max(1, len(test_problems))

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / f'lora_eval_{tag}_aditi_mbppaug.csv', index=False)
    # Checkpoint the aggregate scores so a runtime restart can reload them (see cells 7.3 / 10).
    with open(RESULTS_DIR / f'scores_{tag}_aditi_mbppaug.json', 'w') as f:
        json.dump(scores, f, indent=2)
    print(f'{tag} scores:', {k: round(v, 4) for k, v in scores.items()})
    print('  saved ->', RESULTS_DIR / f'scores_{tag}_aditi_mbppaug.json')
    return scores

### 7.3 — Run **zero-shot** eval first (base model, before any training)

In [ ]:
# 7.3 — zero-shot eval (base model). Resumable: if already computed & saved, just reload it.
import json
_zs_path = RESULTS_DIR / 'scores_zero_shot_aditi_mbppaug.json'
if _zs_path.exists():
    with open(_zs_path) as f:
        zero_shot_scores = json.load(f)
    print('Loaded cached zero-shot scores from', _zs_path, '->', zero_shot_scores)
else:
    zero_shot_scores = evaluate_all(model, tokenizer, test_problems, TASKS, tag='zero_shot')

zero_shot:nl2py:   0%|          | 0/33 [00:00<?, ?it/s]

zero_shot:nl2java:   0%|          | 0/33 [00:00<?, ?it/s]

zero_shot:py2java:   0%|          | 0/33 [00:00<?, ?it/s]

zero_shot:code2nl:   0%|          | 0/33 [00:00<?, ?it/s]

zero_shot scores: {'nl2py': 0.6364, 'nl2java': 0.4848, 'py2java': 0.7273, 'code2nl': 0.2414}
  saved -> /content/drive/MyDrive/lora_finetune/results/scores_zero_shot_aditi_mbppaug.json


---
## ⏸️ END OF PHASE A — RESTART THE RUNTIME HERE (T4 flow)

You have now computed and **saved the zero-shot baseline to Drive**
(`results/scores_zero_shot_aditi_mbppaug.json`). On a **T4, do NOT keep going into training in
this same session** — the base model is still holding VRAM and the (larger, MBPP-augmented)
training run will likely OOM and force you to start over.

**Do this now:**
1. **Runtime → Restart runtime** (clears all VRAM).
2. After restart, re-run **cell 1.1 → cell 6.1** (setup, data, load base model — fast; the model download is cached).
3. **Skip cell 7.3** — or run it, and it will instantly reload the saved zero-shot JSON instead of recomputing.
4. Continue from **cell 8.x** (training) below. If a trained adapter already exists, cell 8.3 will load it and skip training.

*(On an L4 / A100 you can ignore this and run straight through — no restart needed.)*

=== VERIFY after inserting ===
Read the notebook back and confirm: there is now a markdown cell containing the text "END OF PHASE A" positioned AFTER cell `dNj6757R88lD` and BEFORE cell `MQHXonU888lD`. Report PASS/FAIL and the new cell's id. Do not modify any other file or cell.

## 8. QLoRA fine-tuning

Attach a LoRA adapter to the 4-bit base and train on the completion tokens only. The base
weights are frozen, so the zero-shot numbers above remain a valid "before" baseline.

In [ ]:
# 8.1 — tokenize with completion-only label masking
# NOTE: apply_chat_template(tokenize=True) can return tokenizers.Encoding objects
# in some Colab transformers builds, which torch.tensor() can't convert
# ("Could not infer dtype of tokenizers.Encoding"). So we build the chat STRING
# (tokenize=False) and then encode() to plain int lists — the approach proven in
# Vamsi's notebook. Generation (cell 3.2) uses the same chat template, so the
# train-time and eval-time prompt formats match exactly.
def tokenize_example(prompt, target, max_len=TRAIN['max_len']):
      prompt_str = tokenizer.apply_chat_template(
          [{'role': 'user', 'content': prompt}],
          tokenize=False, add_generation_prompt=True)
      prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
      target_ids = tokenizer.encode(target, add_special_tokens=False)
      eos = tokenizer.eos_token_id
      full_ids = prompt_ids + target_ids + [eos]
      if len(full_ids) > max_len:        # drop over-long examples (keeps tensors clean)
          return None
      labels = [-100] * len(prompt_ids) + target_ids + [eos]
      return {'input_ids': full_ids, 'labels': labels}

from torch.utils.data import Dataset
class SFTDataset(Dataset):
      def __init__(self, examples):
          self.data, self.dropped = [], 0
          for p, t in examples:
              ex = tokenize_example(p, t)
              if ex is None:
                  self.dropped += 1
              else:
                  self.data.append(ex)
      def __len__(self):
          return len(self.data)
      def __getitem__(self, i):
          return self.data[i]

train_ds = SFTDataset(train_examples)
print(f'Tokenized training examples: {len(train_ds)} kept, '
        f'{train_ds.dropped} dropped (> max_len={TRAIN["max_len"]})')
# sanity: every field must be a list of plain ints
_e = train_ds[0]
assert all(isinstance(x, int) for x in _e['input_ids']), 'input_ids not plain ints!'
print('input_ids dtype check OK (plain ints).')


Tokenized training examples: 1209 kept, 4 dropped (> max_len=1024)
input_ids dtype check OK (plain ints).


In [ ]:
# 8.2 — collator (pad input_ids with pad_token, labels with -100)
import torch

def collate(batch):
    maxlen = max(len(b['input_ids']) for b in batch)
    pad = tokenizer.pad_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad] * n)
        labels.append(b[
            'labels'] + [-100] * n)
        attn.append([1] * len(b['input_ids']) + [0] * n)
    return {'input_ids': torch.tensor(input_ids),
            'labels': torch.tensor(labels),
            'attention_mask': torch.tensor(attn)}

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
print('torchao removed — no restart needed.')


torchao removed — no restart needed.


In [ ]:
# 8.3 — attach LoRA + train  (resumable: if the adapter is already saved to Drive, load it)
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import TrainingArguments, Trainer

_adapter_ready = (ADAPTER_DIR / 'adapter_config.json').exists()

if _adapter_ready and not isinstance(model, PeftModel):
    # A previous run already trained + saved the adapter to Drive. Reload it and skip training.
    print(f'Found saved adapter at {ADAPTER_DIR} — loading it and SKIPPING training.')
    print('   (To force a fresh retrain, delete that folder first.)')
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
elif isinstance(model, PeftModel):
    print('Model already has a LoRA adapter attached — reusing it (skipping re-wrap).')
else:
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(task_type='CAUSAL_LM', bias='none', **LORA)
    model = get_peft_model(model, lora_cfg)

if not _adapter_ready:
    model.config.use_cache = False  # required with gradient checkpointing
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=str(DATA_DIR / 'lora_runs_aditi'),
        per_device_train_batch_size=TRAIN['batch'],
        gradient_accumulation_steps=TRAIN['grad_accum'],
        num_train_epochs=TRAIN['epochs'],
        learning_rate=TRAIN['lr'],
        warmup_ratio=TRAIN['warmup_ratio'],
        fp16=True, logging_steps=5, save_strategy='no',
        gradient_checkpointing=True, optim='paged_adamw_8bit',
        report_to='none', lr_scheduler_type='cosine')

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
    trainer.train()

    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print('Adapter saved ->', ADAPTER_DIR)
else:
    print('Adapter loaded from disk; no training performed.')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


Step,Training Loss
5,0.442769
10,0.313279
15,0.311403
20,0.250771
25,0.277615
30,0.231417
35,0.225415
40,0.225461
45,0.279889
50,0.160653


Adapter saved -> /content/drive/MyDrive/lora_finetune/lora_adapter_aditi_mbppaug


In [ ]:
# 8.4 — free optimizer memory before eval
import gc, torch
del trainer
gc.collect(); torch.cuda.empty_cache()
model.config.use_cache = True
model.eval()
print('Ready for QLoRA eval.')

Ready for QLoRA eval.


## 9. QLoRA eval (base + adapter)

In [ ]:
qlora_scores = evaluate_all(model, tokenizer, test_problems, TASKS, tag='qlora')

qlora:nl2py:   0%|          | 0/33 [00:00<?, ?it/s]

qlora:nl2java:   0%|          | 0/33 [00:00<?, ?it/s]

qlora:py2java:   0%|          | 0/33 [00:00<?, ?it/s]

qlora:code2nl:   0%|          | 0/33 [00:00<?, ?it/s]

NameError: name 'rougeL_f1' is not defined

In [ ]:
code_only = evaluate_all(model, tokenizer, test_problems, ['code2nl'], tag='qlora_code2nl')
print('code2nl ROUGE-L F1:', round(code_only['code2nl'], 4))


qlora_code2nl:code2nl:   0%|          | 0/33 [00:00<?, ?it/s]

qlora_code2nl scores: {'code2nl': 0.4179}
  saved -> /content/drive/MyDrive/lora_finetune/results/scores_qlora_code2nl_aditi_mbppaug.json
code2nl ROUGE-L F1: 0.4179


In [ ]:
import json
def _load(tag):
    p = RESULTS_DIR / f'scores_{tag}_aditi_mbppaug.json'
    return json.load(open(p)) if p.exists() else {}

merged = {}
merged.update(_load('qlora'))            # any earlier combined save
for t in TASKS:
    merged.update(_load(f'qlora_{t}'))   # per-task files (incl. qlora_code2nl)
missing = [t for t in TASKS if t not in merged]
print('have:', {k: round(v, 4) for k, v in merged.items()})
print('missing:', missing)


have: {'code2nl': 0.4179}
missing: ['nl2py', 'nl2java', 'py2java']


In [ ]:
for t in missing:
    s = evaluate_all(model, tokenizer, test_problems, [t], tag=f'qlora_{t}')
    merged[t] = s[t]
qlora_scores = merged
json.dump(qlora_scores, open(RESULTS_DIR / 'scores_qlora_aditi_mbppaug.json', 'w'), indent=2)
print(qlora_scores)


qlora_nl2py:nl2py:   0%|          | 0/33 [00:00<?, ?it/s]

qlora_nl2py scores: {'nl2py': 0.6364}
  saved -> /content/drive/MyDrive/lora_finetune/results/scores_qlora_nl2py_aditi_mbppaug.json


qlora_nl2java:nl2java:   0%|          | 0/33 [00:00<?, ?it/s]

qlora_nl2java scores: {'nl2java': 0.6061}
  saved -> /content/drive/MyDrive/lora_finetune/results/scores_qlora_nl2java_aditi_mbppaug.json


qlora_py2java:py2java:   0%|          | 0/33 [00:00<?, ?it/s]

qlora_py2java scores: {'py2java': 0.7576}
  saved -> /content/drive/MyDrive/lora_finetune/results/scores_qlora_py2java_aditi_mbppaug.json
{'code2nl': 0.4179398474355802, 'nl2py': 0.6363636363636364, 'nl2java': 0.6060606060606061, 'py2java': 0.7575757575757576}


## 10. Before vs After — comparison table

In [ ]:
# 10 — Before vs After comparison. Robust to runtime restarts: if the score dicts aren't in
# memory (e.g. you restarted between zero-shot and training), reload them from the saved JSON.
import json

def _load_scores(tag):
    p = RESULTS_DIR / f'scores_{tag}_aditi_mbppaug.json'
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

if 'zero_shot_scores' not in globals() or zero_shot_scores is None:
    zero_shot_scores = _load_scores('zero_shot')
if 'qlora_scores' not in globals() or qlora_scores is None:
    qlora_scores = _load_scores('qlora')
assert zero_shot_scores is not None, 'No zero-shot scores in memory or on disk — run cell 7.3 first.'
assert qlora_scores is not None, 'No QLoRA scores in memory or on disk — run cell 9 first.'

TASK_META = {
    'nl2py':   ('NL→Python',  'pass@1'),
    'nl2java': ('NL→Java',    'pass@1'),
    'code2nl': ('Code→NL',    'ROUGE-L F1'),
    'py2java': ('Python→Java', 'pass@1'),
}

rows = []
for t in TASKS:
    label, metric = TASK_META[t]
    zs = zero_shot_scores.get(t, float('nan'))
    ql = qlora_scores.get(t, float('nan'))
    rows.append({'task': label, 'metric': metric, 'n_test': len(test_problems),
                 'zero_shot_7b': round(zs, 4), 'qlora_7b': round(ql, 4),
                 'delta': round(ql - zs, 4)})

comp = pd.DataFrame(rows)
out = RESULTS_DIR / 'lora_comparison_aditi_mbppaug.csv'
comp.to_csv(out, index=False)
print('=== Before vs After (MBPP-augmented): Qwen2.5-Coder-1.5B (Aditi) ===')
print(comp.to_string(index=False))
print('\nSaved ->', out)

=== Before vs After (MBPP-augmented): Qwen2.5-Coder-7B (Aditi) ===
       task     metric  n_test  zero_shot_7b  qlora_7b  delta
  NL→Python     pass@1      33        0.6364    0.6364 0.0000
    NL→Java     pass@1      33        0.4848    0.6061 0.1212
    Code→NL ROUGE-L F1      33        0.2414    0.4179 0.1766
Python→Java     pass@1      33        0.7273    0.7576 0.0303

Saved -> /content/drive/MyDrive/lora_finetune/results/lora_comparison_aditi_mbppaug.csv


## 11. How to run + notes

### Persistent storage (IMPORTANT)
Cell 1.1 now **requires Google Drive** on Colab and writes everything under
`PROJECT_DIR = /content/drive/MyDrive/codegen_week1` (override with the `CODEGEN_PROJECT_DIR`
env var). The adapter, eval CSVs, and score JSONs go to Drive so a runtime restart or crash
does **not** lose them. If the mount fails, the cell raises instead of silently writing to the
ephemeral `/content` — which is what made earlier restarts start from scratch. **You must
complete the Drive authorization popup when cell 1.1 runs.**

Upload `extended_mbpp_seed13.csv` to `{PROJECT_DIR}/results/` before running cell 4.5.

### Recommended run order — two phases with a restart (best for T4)
Restarting between zero-shot and training frees all VRAM, avoiding OOM on the 7B model. The
zero-shot scores are checkpointed to Drive, so the final comparison still works after restart.

**Phase A — zero-shot baseline (no training):**
1. Runtime → Change runtime type → GPU.
2. Run cells 1.1 → 7.3. Cell 7.3 saves `results/scores_zero_shot_aditi_mbppaug.json`.
3. Stop here. **Do not run the training cells yet.**

**Restart:** Runtime → Restart runtime (clears VRAM).

**Phase B — train + evaluate:**
4. Re-run cells 1.1 → 6.1 (setup, data, load base; the model download is cached so it is fast).
   You may skip cell 7.3 — or run it, and it will instantly reload the cached zero-shot JSON.
5. Run cell 8.x (train). The adapter saves to `{PROJECT_DIR}/lora_adapter_mbppaug`.
   If that folder already exists, cell 8.3 **loads it and skips training** (delete it to retrain).
6. Run cell 9 (QLoRA eval) → saves `results/scores_qlora_aditi_mbppaug.json`.
7. Run cell 10 (comparison). It reloads both score JSONs from Drive if not already in memory,
   then prints + saves `results/lora_comparison_aditi_mbppaug.csv`.

### Single-session run (only if you have an L4 / A100)
Just run top-to-bottom: setup → data → load base → 7.3 → 8.x → 9 → 10. No restart needed.

### Fast sanity check first (recommended)
Before the full 7B run, in cell 1.3 set:
```python
os.environ['CODEGEN_MODEL'] = '1.5b'
os.environ['CODEGEN_NTEST'] = '5'
```
Run through once. If the comparison prints end-to-end, switch back to `'7b'` / `'33'`. Delete
the tiny-run artifacts (or use a different `CODEGEN_PROJECT_DIR`) so the 1.5B scores/adapter
are not mistaken for the real ones.

### Outputs (under {PROJECT_DIR})
- `results/lora_comparison_aditi_mbppaug.csv` — headline before/after table
- `results/scores_zero_shot_aditi_mbppaug.json`, `results/scores_qlora_aditi_mbppaug.json` — checkpointed metric dicts (survive restarts)
- `results/lora_eval_zero_shot_aditi_mbppaug.csv`, `results/lora_eval_qlora_aditi_mbppaug.csv` — per-example predictions
- `lora_adapter_mbppaug/` — the trained adapter (reload to skip retraining)

### Comparing with the baseline
Compare the four deltas in `lora_comparison_aditi_mbppaug.csv` against the un-augmented
`results/results.txt`. Watch that Code→NL (not fed MBPP) does not regress.

### Knobs (cell 1.3)
- `TASKS` — the four eval directions.
- `MBPP_AUG_TASKS` — which directions MBPP augments (default nl2py/nl2java/py2java; Code→NL excluded).
- `USE_MBPP_AUG` — master switch for the MBPP augmentation.
- `LORA` / `TRAIN` — adapter rank, epochs, lr (unchanged: r=8, 3 epochs, lr 2e-4).

## 12. Gradio Demo

Interactive demo of the fine-tuned model across all four task directions
(NL→Python, NL→Java, Code→NL, Python→Java). Reuses the prompt builders,
generation function, and extractors defined earlier in this notebook — no
new prompt logic. Run cells 12.1 → 12.5 in order after Section 8/9 have
been run at least once (so an adapter exists at `ADAPTER_DIR`).

In [ ]:
# 12.1 — install gradio (self-contained: doesn't depend on the `sh` helper from cell 1.2)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'gradio'], check=False)
print('gradio installed.')

gradio installed.


In [ ]:
# 12.2 — ensure an inference-ready model (base + LoRA adapter attached)
# Fully self-contained: reconstructs config/tokenizer/model if this is a fresh
# kernel that skipped straight to Section 12, or reuses what's already in
# memory if run right after training (Section 8/9).
import os, subprocess, sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# peft's is_torchao_available() actively version-gates: it raises ImportError
# if torchao is installed but below 0.16.0 (Colab ships 0.10.0), rather than
# just returning False. Uninstalling doesn't help once peft is already
# imported in this kernel (e.g. from an earlier failed attempt, or Section
# 8/9 having run) — the reliable fix is upgrading torchao to a version that
# satisfies the check.
if 'peft' in sys.modules:
    print('peft is already imported in this kernel — if the upgrade below does not '
          'fix the ImportError, Runtime > Restart runtime and rerun 12.1 -> 12.2.')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'torchao>=0.17.0'], check=False)
from peft import PeftModel

if 'PROJECT_DIR' not in globals():
    try:
        import google.colab  # noqa: F401
        PROJECT_DIR = Path(os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/lora_finetune'))
    except Exception:
        PROJECT_DIR = Path('.').resolve()
if 'MODEL_SIZE' not in globals():
    MODEL_SIZE = os.environ.get('CODEGEN_MODEL', '1.5b')
if 'MODEL_ID' not in globals():
    MODELS = {'1.5b': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', '7b': 'Qwen/Qwen2.5-Coder-7B-Instruct'}
    MODEL_ID = MODELS[MODEL_SIZE]
if 'ADAPTER_DIR' not in globals():
    DATA_DIR = Path(os.environ.get('CODEGEN_DATA_DIR', str(PROJECT_DIR)))
    ADAPTER_DIR = DATA_DIR / 'lora_adapter_mbppaug'

if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise RuntimeError(
        f'No adapter found at {ADAPTER_DIR}. Run Section 8 (training) first.')

if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

if 'model' in globals() and isinstance(model, PeftModel):
    print('Using the already-attached LoRA adapter from this session.')
else:
    print(f'Loading base model + adapter from {ADAPTER_DIR} ...')
    if MODEL_SIZE == '7b':
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

model.eval()
print('Model ready for demo inference.')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready for demo inference.


In [ ]:
# 12.3 — inference wrappers per task (reuse existing prompt builders / extractors)
def demo_nl2py(nl, signature):
    raw = generate_text(tokenizer, model, p_nl2py(nl, signature), 512)
    return extract_python_body(raw)

def demo_nl2java(nl, java_decl):
    raw = generate_text(tokenizer, model, p_nl2java(nl, java_decl), 900)
    code = extract_java_body(raw)
    return f'```java\n{code}\n```'

def demo_code2nl(py_code):
    raw = generate_text(tokenizer, model, p_code2nl(py_code), 400)
    return strip_markdown(raw)

def demo_py2java(py_code, java_decl):
    raw = generate_text(tokenizer, model, p_py2java(py_code, java_decl), 900)
    code = extract_java_body(raw)
    return f'```java\n{code}\n```'

print('Demo inference wrappers ready: demo_nl2py, demo_nl2java, demo_code2nl, demo_py2java')

Demo inference wrappers ready: demo_nl2py, demo_nl2java, demo_code2nl, demo_py2java


In [ ]:
# 12.4 — Gradio Blocks UI: one tab per task
import gradio as gr

with gr.Blocks(title='QLoRA Code-Intelligence Demo') as demo:
    gr.Markdown(f'# QLoRA Fine-Tuned {MODEL_ID} — Code Intelligence Demo')

    with gr.Tabs():
        with gr.Tab('NL → Python'):
            nl_in = gr.Textbox(label='Specification (natural language)', lines=4)
            sig_in = gr.Textbox(label='Required Python signature', value='def solve(x):')
            btn = gr.Button('Generate')
            out = gr.Code(label='Generated Python', language='python')
            btn.click(demo_nl2py, inputs=[nl_in, sig_in], outputs=out)

        with gr.Tab('NL → Java'):
            nl_in2 = gr.Textbox(label='Specification (natural language)', lines=4)
            decl_in = gr.Textbox(label='Required Java declaration',
                                  value='public class Solution { public static int solve(int x) {')
            btn2 = gr.Button('Generate')
            out2 = gr.Markdown(label='Generated Java')
            btn2.click(demo_nl2java, inputs=[nl_in2, decl_in], outputs=out2)

        with gr.Tab('Code → NL'):
            # gr.Code renders invisibly as an input in this Gradio build, so use a
            # plain multi-line Textbox for pasting the Python function instead.
            code_in = gr.Textbox(label='Python function', lines=10,
                                 placeholder='Paste a Python function here...')
            btn3 = gr.Button('Generate')
            out3 = gr.Textbox(label='Generated specification', lines=4)
            btn3.click(demo_code2nl, inputs=[code_in], outputs=out3)

        with gr.Tab('Python → Java'):
            py_in = gr.Textbox(label='Python function', lines=10,
                               placeholder='Paste a Python function here...')
            decl_in2 = gr.Textbox(label='Required Java declaration',
                                   value='public class Solution { public static int solve(int x) {')
            btn4 = gr.Button('Generate')
            out4 = gr.Markdown(label='Generated Java')
            btn4.click(demo_py2java, inputs=[py_in, decl_in2], outputs=out4)

print('Gradio Blocks UI defined.')

Gradio Blocks UI defined.


In [ ]:
# 12.5 — launch via ngrok tunnel
# Gradio's built-in share=True tunnel (frpc/cloudflared) failed to establish on
# this Colab network ("Could not create share link"), so instead we run the
# Gradio server locally (share=False) and tunnel it through ngrok, which works
# reliably across firewalled/restrictive networks.
#
# Before running this cell: sign up free at https://ngrok.com, then copy your
# authtoken from https://dashboard.ngrok.com/get-started/your-authtoken and
# paste it into NGROK_AUTHTOKEN below.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'pyngrok'], check=False)

from pyngrok import ngrok

NGROK_AUTHTOKEN = '3GffkwCakxigVKMFoOft5kBzeDB_FoaoQ5J6oeCoztzBNZix'  # <-- paste your ngrok authtoken here
if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        'Set NGROK_AUTHTOKEN above to your ngrok authtoken '
        '(https://dashboard.ngrok.com/get-started/your-authtoken) before running this cell.')
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Close any previous Gradio server + ngrok tunnels (free ngrok accounts allow
# only one tunnel at a time, so a stale tunnel from an earlier run will block
# a new one from opening).
try:
    demo.close()
except Exception:
    pass
for _tunnel in ngrok.get_tunnels():
    ngrok.disconnect(_tunnel.public_url)

# Don't hardcode a port: a Gradio server from a previous run may still hold it
# ("Cannot find empty port in range"). Let Gradio auto-scan for any free port,
# then tunnel ngrok to whichever port it actually bound to.
demo.launch(share=False, debug=False, prevent_thread_lock=True)

public_tunnel = ngrok.connect(demo.server_port, 'http')
print('Public URL (ngrok):', public_tunnel.public_url)
print('(Gradio bound to local port', demo.server_port, ')')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Public URL (ngrok): https://penalty-duckbill-elude.ngrok-free.dev
(Gradio bound to local port 7861 )
